# Clasificación con Gemini — Derecho a las Víctimas

**Proyecto CIAI**

Clasificación de las 62 normas del test set usando Gemini via Google Colab AI.
Este notebook usa `google.colab.ai` que no tiene el límite de 20 requests/día de la API estándar.

**Modelo:** gemini-2.5-flash-lite  
**Estrategia:** few-shot con 9 ejemplos  
**Input:** Título + Resumen + Artículos (hasta 500 caracteres)  



## 1. Instalación

In [1]:
!pip install -q openpyxl

## 2. Imports

In [ ]:
import pandas as pd
import time
from google.colab import ai, files

# Modelo disponible en Colab AI
MODELO = 'google/gemini-2.5-flash-lite'

# Pausa entre llamadas (segundos)
PAUSA = 3

print('✅ Imports OK')
print('Modelos disponibles:')
ai.list_models()

✅ Imports OK
Modelos disponibles:


## 3. Prompt de clasificación

Basado en el criterio definido por los investigadores. Few-shot con 6 ejemplos.

In [ ]:
 PROMPT_SISTEMA =  """Sos un experto en análisis de normativa legal argentina.
Tu tarea es clasificar si una norma consagra o no derechos de las víctimas.

CRITERIO:
- caso_ok = 1 (SÍ consagra derechos de víctimas): la norma establece derechos, garantías u obligaciones
  relativas a personas consideradas víctimas. Incluye mecanismos de denuncia, acompañamiento, asistencia,
  reparación/indemnización, o define organigramas de organismos directamente vinculados a derechos de víctimas.

- caso_ok = 0 (NO consagra derechos de víctimas): la norma menciona la palabra "víctima" pero legisla
  principalmente sobre otros sujetos (victimarios, operadores judiciales, testigos), o solo reorganiza
  estructuras administrativas, o usa el término como contexto sin involucrar directamente a las víctimas.

EJEMPLOS:
- Ley 1487 (Subsidio a víctimas de inundaciones) → 1: Otorga asistencia directa a víctimas.
- Decreto 9088 (Subsidio a deudos de víctimas de accidente) → 1: Reparación económica directa.
- Decreto 558 (Pago de sentencia CIDH a víctimas) → 1: Ejecuta reparación concreta a víctimas.
- Ley 340 (Código Civil) → 0: Legisla sobre responsabilidad civil general, no sobre víctimas.
- Decreto 851 (Organigrama Programa Verdad y Justicia) → 0: Solo define estructura administrativa.
- Ley 25633 (Día Nacional de la Memoria) → 0: Declaración conmemorativa sin derechos concretos.
- Ley 27403 (Acuerdo Marco Argentina-Bolivia sobre trata) → 0: Aprueba un acuerdo de cooperación entre estados, no establece derechos concretos para víctimas en el ordenamiento argentino.
- Ley 23097 (Código Penal — modificación Art. 144) → 0: Modifica normas penales sobre conductas de victimarios, no establece derechos para las víctimas.
- Decreto 1755 (Ministerio de Justicia — crea Programa de Protección a Víctimas) → 0: Define estructura administrativa del ministerio, no establece derechos sustantivos para las víctimas.

INSTRUCCIÓN:
Respondé con el número 1 o 0 seguido de dos puntos y una oración breve explicando tu decisión.
Ejemplo: '1: La norma establece asistencia directa a víctimas de inundación.'"""
print('✅ Prompt configurado')

## 4. Carga de datos

In [ ]:
df = pd.read_excel('test.xlsx')
df['Número'] = df['Número'].astype(str).str.strip()

print(f'Normas a clasificar: {len(df)}')
print(f'Columnas disponibles: {df.columns.tolist()}')
df.head(3)

Normas a clasificar: 62
Columnas disponibles: ['Tipo', 'Número', 'Añosanción', 'Título', 'Resumen', 'Artículos']


,Tipo,Número,Añosanción,Título,Resumen,Artículos
0,Ley,5209,1907,EMERGENCIA PéBLICA,SOCORROS PARA LAS VICTIMAS DE LA INUNDACION EN...,NaN
1,Ley,26004,2004,ACUERDOS,APRUEBASE EL ACUERDO DE ASISTENCIA JURIDICA MU...,Art. 23: 2.- Los Estados Parte se prestarán as...
2,Ley,25815,2003,CODIGO PENAL. Modificación del mismo. Sustitúy...,NaN,ARTICULO 1º — Modifícase el artículo 23 del Có...


## 5. Clasificación

In [ ]:
def construir_prompt_usuario(row):
    titulo    = str(row.get('Título', '')).strip()    if pd.notna(row.get('Título'))    else ''
    resumen   = str(row.get('Resumen', '')).strip()   if pd.notna(row.get('Resumen'))   else ''
    articulos = str(row.get('Artículos', '')).strip() if pd.notna(row.get('Artículos')) else ''

    texto = f"Tipo: {row.get('Tipo', '')}\nNúmero: {row.get('Número', '')}\nTítulo: {titulo}"
    if resumen:
        texto += f"\nResumen: {resumen}"
    if articulos:
        texto += f"\nArtículos relevantes: {articulos[:500]}"
    texto += "\n\n¿Esta norma consagra derechos de las víctimas?"
    return texto


def parsear_respuesta(respuesta):
    respuesta = str(respuesta).strip()
    if respuesta.startswith('1'):
        return 1, respuesta
    elif respuesta.startswith('0'):
        return 0, respuesta
    else:
        return -1, f"Respuesta inválida: {respuesta}"


predicciones   = []
justificaciones = []

for i, row in df.iterrows():
    prompt = PROMPT_SISTEMA + '\n\n' + construir_prompt_usuario(row)
    try:
        respuesta = ai.generate_text(prompt, model_name=MODELO)
        pred, just = parsear_respuesta(respuesta)
    except Exception as e:
        pred = -1
        just = f"Error: {e}"

    predicciones.append(pred)
    justificaciones.append(just)
    print(f"  [{i+1}/{len(df)}] Norma {row.get('Número', '')} → {pred}")
    time.sleep(PAUSA)

print('\n✅ Clasificación terminada')

  [1/62] Norma 5209 → 1
  [2/62] Norma 26004 → 1
  [3/62] Norma 25815 → 0
  [4/62] Norma 25632 → 1
  [5/62] Norma 844 → 1
  [6/62] Norma 1486 → 0
  [7/62] Norma 26690 → 1
  [8/62] Norma 2767 → 1
  [9/62] Norma 13116 → 1
  [10/62] Norma 1318 → 1
  [11/62] Norma 14414 → 1
  [12/62] Norma 18577 → 1
  [13/62] Norma 21507 → 1
  [14/62] Norma 606 → 0
  [15/62] Norma 26958 → 0
  [16/62] Norma 4552 → 1
  [17/62] Norma 1982 → 0
  [18/62] Norma 2009 → 0
  [19/62] Norma 24946 → 0
  [20/62] Norma 235 → 1
  [21/62] Norma 25742 → 1
  [22/62] Norma 26378 → 1
  [23/62] Norma 27149 → 0
  [24/62] Norma 23280 → 0
  [25/62] Norma 525 → 1
  [26/62] Norma 231 → 0
  [27/62] Norma 174 → 0
  [28/62] Norma 26842 → 1
  [29/62] Norma 812 → 1
  [30/62] Norma 2637 → 1
  [31/62] Norma 27508 → 1
  [32/62] Norma 888 → 0
  [33/62] Norma 636 → 1
  [34/62] Norma 25616 → 1
  [35/62] Norma 1800 → 1
  [36/62] Norma 26827 → 1
  [37/62] Norma 27402 → 1
  [38/62] Norma 25112 → 1
  [39/62] Norma 840 → 0
  [40/62] Norma 23984 → 

## 6. Resultados y guardado

In [ ]:
df_resultado = df[['Tipo', 'Número', 'Título']].copy()
df_resultado['caso_ok']       = predicciones
df_resultado['Justificación'] = justificaciones

validos = [p for p in predicciones if p != -1]
errores = [p for p in predicciones if p == -1]

print(f'Total clasificadas: {len(validos)} de {len(df)}')
print(f'caso_ok = 1: {predicciones.count(1)}')
print(f'caso_ok = 0: {predicciones.count(0)}')
if errores:
    print(f'Errores (-1): {len(errores)}')

# Guardar
df_resultado.to_excel('resultado_gemini_colab_prompt2.xlsx', index=False)
print('\n✅ Guardado: resultado_gemini_colab_prompt2.xlsx')

df_resultado

Total clasificadas: 62 de 62
caso_ok = 1: 40
caso_ok = 0: 22

✅ Guardado: resultado_gemini_colab_prompt2.xlsx


,Tipo,Número,Título,caso_ok,Justificación
0,Ley,5209,EMERGENCIA PéBLICA,1,1: La norma establece socorros directos para l...
1,Ley,26004,ACUERDOS,1,1: La norma establece asistencia en procedimie...
2,Ley,25815,CODIGO PENAL. Modificación del mismo. Sustitúy...,0,0: La norma se enfoca en el decomiso de bienes...
3,Ley,25632,CONVENCIONES,1,1: La norma establece mecanismos para la indem...
4,Decreto Reglamentario,844,FONDO DE ASISTENCIA DIRECTA A VICTIMAS DE TRATA,1,1: La norma reglamenta un fondo de asistencia ...
...,...,...,...,...,...
57,Ley,24417,PROTECCION CONTRA LA VIOLENCIA FAMILIAR,1,1: La norma establece el derecho de toda perso...
58,Ley,6285,SUBSIDIO ESTATAL,1,1: La norma establece subsidios directos a víc...
59,Ley,27146,JUSTICIA,0,0: La norma promueve la claridad en el lenguaj...
60,Ley,25763,DERECHOS DEL NIÑO,1,1: La norma establece medidas de protección y ...


## 7. Descargar resultado

In [ ]:
files.download('resultado_gemini_colab_prompt2.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>